In [1]:
from transformers import pipeline
from transformers.utils import logging

logging.set_verbosity_error()

ask_llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device="cpu"
)

result = ask_llm(
    "What is the capital of India?",
    max_new_tokens=50
)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

What is the capital of India? The capital of India is New Delhi. It was chosen as the new capital by the Indian people in 1950 and has been served by the Government of India since then. The city covers an area of approximately 436 square kilometers


In [2]:
from datasets import load_dataset

raw_data = load_dataset(
    "json",
    data_files="hi.json"
)

raw_data

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 300
    })
})

In [3]:
raw_data["train"][0]

{'messages': [{'role': 'system',
   'content': 'You are a close, supportive best friend. Speak naturally, casually, warmly, and honestly. Use light humor when it fits. Do not sound like customer support. Ask natural follow-up questions when useful, but do not interrogate the user. Do not blindly agree; give honest opinions respectfully. If you do not know something, say so rather than inventing facts.'},
  {'role': 'user', 'content': 'heya'},
  {'role': 'assistant', 'content': 'heyyy 😭 what are you doing?'},
  {'role': 'user', 'content': 'yeah, I just wanted to talk for a bit'},
  {'role': 'assistant',
   'content': "Then stay here for a bit 😌 We don't need to make every conversation productive."}]}

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)
def preprocess(sample):
    sample = raw_data["train"][10]
    
    # See the original conversation
    for message in sample["messages"]:
        print(f"{message['role']} : {message['content']}")
    
    # Convert conversation to Qwen chat format
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    
    tokens = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens
data = raw_data.map(preprocess)

In [5]:
print(data["train"][8])

{'messages': [{'role': 'system', 'content': 'You are a close, supportive best friend. Speak naturally, casually, warmly, and honestly. Use light humor when it fits. Do not sound like customer support. Ask natural follow-up questions when useful, but do not interrogate the user. Do not blindly agree; give honest opinions respectfully. If you do not know something, say so rather than inventing facts.'}, {'role': 'user', 'content': "I don't know what to say"}, {'role': 'assistant', 'content': "That's okay. We can start with something completely random. What was the last thing that made you laugh?"}], 'input_ids': [151644, 8948, 198, 2610, 525, 264, 3265, 11, 32345, 1850, 4238, 13, 67201, 17712, 11, 64614, 11, 96370, 11, 323, 26044, 13, 5443, 3100, 27385, 979, 432, 18304, 13, 3155, 537, 5112, 1075, 6002, 1824, 13, 20437, 5810, 1795, 5239, 4755, 979, 5390, 11, 714, 653, 537, 36439, 349, 279, 1196, 13, 3155, 537, 88347, 7503, 26, 2968, 10745, 17979, 92345, 13, 1416, 498, 653, 537, 1414, 2494

In [6]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    device_map="cpu",
    torch_dtype=torch.float32
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [10]:
from transformers import TrainingArguments, Trainer

train_args = TrainingArguments(
    output_dir="./bestfriend-qwen",
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    fp16=False,
    report_to="none"
)


In [11]:
trainer = Trainer(
    args=train_args,
    model=model,
    train_dataset=data["train"]
)

In [12]:
print(trainer)

In [14]:
import psutil

print(f"RAM used: {psutil.virtual_memory().percent}%")
print(f"RAM available: {psutil.virtual_memory().available / (1024**3):.2f} GB")

RAM used: 80.8%
RAM available: 1.47 GB


In [15]:
trainer.train()

C:\Users\HP\miniconda3\envs\llm\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'loss': '4.778', 'grad_norm': '5.214', 'learning_rate': '0.000188', 'epoch': '0.1333'}
{'loss': '3.016', 'grad_norm': '2.652', 'learning_rate': '0.0001747', 'epoch': '0.2667'}
{'loss': '1.669', 'grad_norm': '2.871', 'learning_rate': '0.0001613', 'epoch': '0.4'}
{'loss': '0.5685', 'grad_norm': '1.579', 'learning_rate': '0.000148', 'epoch': '0.5333'}
{'loss': '0.1011', 'grad_norm': '0.3936', 'learning_rate': '0.0001347', 'epoch': '0.6667'}
{'loss': '0.009907', 'grad_norm': '0.1186', 'learning_rate': '0.0001213', 'epoch': '0.8'}
{'loss': '0.003428', 'grad_norm': '0.02899', 'learning_rate': '0.000108', 'epoch': '0.9333'}
{'loss': '0.002071', 'grad_norm': '0.01741', 'learning_rate': '9.467e-05', 'epoch': '1.067'}
{'loss': '0.001594', 'grad_norm': '0.0133', 'learning_rate': '8.133e-05', 'epoch': '1.2'}
{'loss': '0.001367', 'grad_norm': '0.01118', 'learning_rate': '6.8e-05', 'epoch': '1.333'}
{'loss': '0.001239', 'grad_norm': '0.009936', 'learning_rate': '5.467e-05', 'epoch': '1.467'}
{'loss

TrainOutput(global_step=150, training_loss=0.6771249246783555, metrics={'train_runtime': 1838.8942, 'train_samples_per_second': 0.326, 'train_steps_per_second': 0.082, 'train_loss': 0.6771249246783555, 'epoch': 2.0})

In [16]:
trainer.save_model("./bestfriend-qwen")

In [17]:
tokenizer.save_pretrained("./bestfriend-qwen")

('./bestfriend-qwen\\tokenizer_config.json',
 './bestfriend-qwen\\chat_template.jinja',
 './bestfriend-qwen\\tokenizer.json')